# 01 — Análise Exploratória (EDA)

Exploração inicial da base de tweets em pt-BR. Lógica de produção vive em `src/`; aqui apenas exploramos.

> **Atenção ao *leakage*:** os rótulos vêm de emoticons/hashtags (`query_used`) presentes no próprio texto. A limpeza (`src/preprocessing`) os remove antes da modelagem.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

from src.config.paths import get_paths
from src.data.loader import load_raw_tweets
from src.preprocessing.cleaning import clean_tweet

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
rng.normal()
plt.rcParams["figure.dpi"] = 120
sns.set_theme(style="whitegrid")

In [ ]:
paths = get_paths()
df = load_raw_tweets(paths.raw_labeled, sample_size=50_000, seed=RANDOM_SEED)
df.head()

## Distribuição de sentimento

In [ ]:
dist = df.group_by("sentiment").len().sort("len", descending=True)
dist.head()

## Efeito da limpeza (antes x depois)

In [ ]:
amostra = df.get_column("tweet_text").head(5).to_list()
for texto in amostra:
    print("BRUTO :", texto)
    print("LIMPO :", clean_tweet(texto))
    print("-" * 60)

In [ ]:
comprimentos = (
    df.with_columns(
        pl.col("tweet_text")
        .map_elements(lambda t: len(clean_tweet(t)), return_dtype=pl.Int64)
        .alias("len_limpo")
    )
    .get_column("len_limpo")
    .to_numpy()
)
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(comprimentos, bins=40, ax=ax)
ax.set_title("Distribuição do comprimento do texto limpo")
ax.set_xlabel("Nº de caracteres")
ax.set_ylabel("Frequência")
plt.tight_layout()